# Evaluation, Bootstrap CIs, and Paired Deltas

Compute bootstrap 95 % confidence intervals on per-(model, emotion) F1, plus paired deltas between every model pair. No training; reads prediction CSVs written by steps 02 (classical + transformer), 03 (weak-zero MTL), and 04 (masked MTL).

Wraps `scripts/evaluation_bootstrap.py`. Runs on CPU in roughly a minute at 1,000 bootstrap iterations.

## 0. Install / check packages

In [ ]:
import importlib.util
for pkg in ('numpy', 'pandas', 'sklearn'):
    print(f"{pkg:14s}: {'OK' if importlib.util.find_spec(pkg) else 'MISSING'}")

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

## 2. Run paired bootstrap

In [ ]:
!python scripts/evaluation_bootstrap.py --n-boot 1000 --seed 42

## 3. Per-(model, emotion) F1 with 95 % bootstrap CI

In [ ]:
import pandas as pd
out = REPO_ROOT / '05_EvaluationBootstrap' / 'outputs'
ci = pd.read_csv(out / '05_bootstrap_ci.csv')
ci.pivot(index='emotion', columns='model', values='f1').round(3)

## 4. Macro F1 summary across models

In [ ]:
pd.read_csv(out / '05_macro_summary.csv').round(3)

## 5. Significant paired deltas (95 % CI excludes 0)

In [ ]:
deltas = pd.read_csv(out / '05_paired_deltas.csv')
sig = deltas[deltas['sig_95'] == True].copy()
print(f'{len(sig)} / {len(deltas)} paired deltas are significant at 95 % CI')
sig.round(3).head(30)